# 02. 크롤링 파이프라인 — 데이터 확인 및 Logic App 수동 실행

이 노트북에서는:
1. Blob Storage에 크롤링된 파일이 몇개인지 확인합니다.
2. 만약 아무 데이터도 없다면 Logic App을 수동으로 실행합니다.
3. Logic App 상태와 최근 실행 이력을 확인합니다.


## 1. 환경 설정

In [ ]:
import os
import sys
import json
import subprocess
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

CRAWL_URL    = os.environ.get('AZURE_FUNCTION_CRAWL_URL', '')
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
CONTAINER    = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
RG_NAME      = os.environ.get('AZURE_RESOURCE_GROUP', 'rg-rag-indexing-lab-swc')
LOGIC_APP    = os.environ.get('AZURE_LOGIC_APP_NAME', '')

print(f"Function URL : {CRAWL_URL}")
print(f"Storage      : {STORAGE_NAME}")
print(f"Container    : {CONTAINER}")
print(f"Logic App    : {LOGIC_APP}")

## 2. Blob Storage 크롤링 결과 확인

현재 저장된 크롤링 데이터의 파일 개수를 확인합니다.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(CONTAINER)

blobs = list(container_client.list_blobs())

# Blob 경로: {source}/{date}/{source}_{seq}.json
sources = sorted(set(b.name.split('/')[0] for b in blobs if '/' in b.name))

print(f"\n{'='*60}")
print(f"크롤링 데이터 현황")
print(f"{'='*60}")
print(f"총 {len(blobs):,}개 파일 | {len(sources)}개 소스 폴더\n")

if len(blobs) == 0:
    print("⚠️  아직 크롤링된 데이터가 없습니다.")
    print("다음 단계에서 Logic App을 수동으로 실행하세요.\n")
else:
    for src in sources:
        src_blobs = [b for b in blobs if b.name.startswith(f"{src}/")]
        dates = sorted(set(b.name.split('/')[1] for b in src_blobs if b.name.count('/') >= 2), reverse=True)
        print(f"  /{src}/")
        print(f"    전체: {len(src_blobs)}개 파일, {len(dates)}개 날짜")
        for d in dates[:3]:
            cnt = sum(1 for b in src_blobs if f"{src}/{d}/" in b.name)
            print(f"      {d}: {cnt}개")
        print()

In [ ]:
# 첫 번째 소스의 최신 JSON 파일 미리보기
if sources:
    src = sources[0]
    json_blobs = sorted(
        [b for b in blobs if b.name.startswith(f"{src}/") and b.name.endswith('.json')],
        key=lambda b: b.name,
        reverse=True,
    )
    if json_blobs:
        blob = container_client.get_blob_client(json_blobs[0].name)
        content = blob.download_blob().readall().decode('utf-8')
        doc = json.loads(content)
        print(f"\n[샘플] {json_blobs[0].name}\n")
        print(json.dumps(doc, ensure_ascii=False, indent=2)[:1500])
        if len(json.dumps(doc, ensure_ascii=False, indent=2)) > 1500:
            print("\n... (생략)")
    else:
        print(f"\n/{src}/ 폴더에 .json 파일 없음")

## 3. 크롤링 데이터가 없으면 Logic App 수동 실행

아래 셀을 실행하면 크롤링이 즉시 시작됩니다.

In [ ]:
import requests

if len(blobs) == 0:
    if not CRAWL_URL:
        print("❌ AZURE_FUNCTION_CRAWL_URL 미설정 — .env 파일을 확인하세요")
    else:
        payload = {"limit": 10, "triggered_by": "notebook-manual"}
        print("\n크롤링 작업 시작 요청 전송...")
        try:
            resp = requests.post(CRAWL_URL, json=payload, timeout=(5, 20))
            print(f"HTTP Status: {resp.status_code}")

            is_failed = resp.status_code >= 400
            body = {}
            try:
                body = resp.json() if resp.content else {}
                if isinstance(body, dict):
                    if body.get("success") is False or body.get("failed") in (True, 1):
                        is_failed = True
                    if body.get("error") or body.get("exception"):
                        is_failed = True
            except ValueError:
                body = {}

            if is_failed:
                print("❌ 시작 단계에서 실패 신호가 감지되었습니다.")
                if body:
                    print(json.dumps(body, ensure_ascii=False, indent=2))
            else:
                print("✅ 크롤링 시작 요청 정상 접수")
                if body:
                    print(json.dumps(body, ensure_ascii=False, indent=2))
                print("\n💡 배경에서 크롤링이 진행 중입니다. 완료 대기는 하지 않습니다.")

        except requests.exceptions.ReadTimeout:
            print("⏱️ 응답 대기 시간 초과 — 요청은 전달되었을 가능성이 높습니다.")
        except requests.RequestException as e:
            print(f"❌ 요청 전송 실패: {e}")
else:
    print(f"✅ 이미 {len(blobs)}개 파일이 저장되어 있으므로 수동 실행을 생략합니다.")

## 4. Logic App 상태 확인

In [ ]:
workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    discover_cmd = [
        "az", "logic", "workflow", "list",
        "--resource-group", RG_NAME,
        "--query", "[].name",
        "-o", "tsv",
    ]
    discover_res = subprocess.run(discover_cmd, capture_output=True, text=True)
    if discover_res.returncode == 0:
        names = [n.strip() for n in discover_res.stdout.splitlines() if n.strip()]
        if len(names) == 1:
            workflow_name = names[0]
            print(f"\n자동 선택: {workflow_name}\n")
        elif len(names) > 1:
            print(f"\n여러 워크플로우 발견: {names}")
            print(".env에서 AZURE_LOGIC_APP_NAME을 명시하세요.\n")
            workflow_name = None
        else:
            print(f"\n리소스 그룹 '{RG_NAME}'에 Logic App 워크플로우가 없습니다.\n")
            workflow_name = None

if workflow_name:
    !az logic workflow show \
        --name {workflow_name} \
        --resource-group {RG_NAME} \
        --query \"{name:name, state:properties.state, location:location}\" \
        --output json 2>/dev/null | python -m json.tool || echo "조회 실패"

## 5. 최근 실행 이력

Logic App의 최근 실행 상태와 실패 건수를 확인합니다.

In [ ]:
if workflow_name:
    sub_cmd = ["az", "account", "show", "--query", "id", "-o", "tsv"]
    sub_res = subprocess.run(sub_cmd, capture_output=True, text=True)
    if sub_res.returncode == 0 and sub_res.stdout.strip():
        sub_id = sub_res.stdout.strip()
        url = (
            f"https://management.azure.com/subscriptions/{sub_id}"
            f"/resourceGroups/{RG_NAME}"
            f"/providers/Microsoft.Logic/workflows/{workflow_name}/runs"
            f"?api-version=2019-05-01"
        )
        run_cmd = [
            "az", "rest",
            "--method", "get",
            "--url", url,
            "--query", "{total:length(value), succeeded:length(value[?properties.status=='Succeeded']), failed:length(value[?properties.status=='Failed']), latest_status:value[0].properties.status, latest_time:value[0].properties.startTime}",
            "-o", "json",
        ]
        run_res = subprocess.run(run_cmd, capture_output=True, text=True)
        if run_res.returncode == 0:
            data = json.loads(run_res.stdout)
            total = data.get("total", 0)
            succeeded = data.get("succeeded", 0)
            failed = data.get("failed", 0)
            latest_status = data.get("latest_status", "Unknown")
            latest_time = data.get("latest_time", "N/A")

            print(f"\n{'='*60}")
            print(f"Logic App 실행 이력")
            print(f"{'='*60}")
            print(f"총 실행   : {total}건")
            print(f"성공     : {succeeded}건")
            print(f"실패     : {failed}건")
            print(f"최근 상태: {latest_status}")
            print(f"최근 실행: {latest_time}")
            print()
            if failed > 0:
                print(f"❌ 최근 실행 중 {failed}건 실패가 감지되었습니다.")
                print("   Azure Portal에서 실행 이력 상세를 확인하세요.")
            else:
                print("✅ 최근 실행에서 실패가 감지되지 않았습니다.")
        else:
            print("실행 이력 조회 실패 (권한 확인 필요)")